In [ ]:
import pandas as pd

def backtest_trades(price_data, signal_data, entry_time='09:00', tp=0.05, sl=0.05):
    """
    Backtest trading signals on BTC price data.

    Parameters:
    price_data (pd.DataFrame): DataFrame containing BTC price data with columns ['Datetime', 'Open', 'High', 'Low', 'Close'].
    signal_data (pd.DataFrame): DataFrame containing trading signals with columns ['Datetime', 'Signal'].
    entry_time (str): Time of day to open positions. Format 'HH:MM'.
    tp (float): Take profit as a percentage of the entry price.
    sl (float): Stop loss as a percentage of the entry price.

    Returns:
    pd.DataFrame: DataFrame containing the backtest results with columns ['Datetime', 'Side', 'Result', 'Duration'].
    """
    # Initialize the output dataframe
    output_data = pd.DataFrame(columns=['Datetime', 'Side', 'Result', 'Duration'])
    
    # Ensure the datetime columns are in datetime format
    price_data['datetime'] = pd.to_datetime(price_data['datetime'])
    signal_data['datetime'] = pd.to_datetime(signal_data['datetime'])
    
    # Iterate through signal_data
    for i, row in signal_data.iterrows():
        signal_datetime = row['datetime']
        signal_value = row['Signal']
        
        # Determine the entry action based on the signal value
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        # Find the entry price based on the specified entry time
        entry_datetime = signal_datetime.replace(hour=int(entry_time.split(':')[0]), minute=int(entry_time.split(':')[1]))
        entry_price_row = price_data.loc[price_data['datetime'] == entry_datetime]
        if entry_price_row.empty:
            continue
        entry_price = entry_price_row['close'].values[0]
        
        # Calculate TP and SL prices
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        # Initialize result and duration
        result = 0
        duration = 0
        
        # Check subsequent prices for TP or SL hit
        subsequent_prices = price_data.loc[price_data['datetime'] > entry_datetime]
        for j, price_row in subsequent_prices.iterrows():
            if side == 'Buy':
                if price_row['high'] >= tp_price:
                    result = 1
                    break
                elif price_row['low'] <= sl_price:
                    result = -1
                    break
            else:
                if price_row['low'] <= tp_price:
                    result = 1
                    break
                elif price_row['high'] >= sl_price:
                    result = -1
                    break
            duration += 1
        
        # Append results to output dataframe
        output_data = output_data.append({
            'Datetime': signal_datetime,
            'Side': side,
            'Result': result,
            'Duration': duration
        }, ignore_index=True)
    
    # Save the results to CSV files month by month
    for name, group in output_data.groupby(output_data['Datetime'].dt.to_period('M')):
        group.to_csv(f'results_{name}.csv', index=False)

    return output_data


In [ ]:
price_data = pd.read_csv('E:\SignalModel\price 2024-05-01, 2024-06-01 min.csv', parse_dates=['datetime'])
signal_data = pd.read_csv('signal_data.csv', parse_dates=['datetime'])

results = backtest_trades(price_data, signal_data, entry_time='09:00', tp=0.05, sl=0.05)
